## 1. Calculation of colors - Settings

**Requirements:** `requests` and `astropy`, both already available in Google Colab. If running the notebook locally, install them with pip install requests astropy.

In [5]:
# 
# %pip install "astropy" "requests"
#


In [7]:
import re
import warnings
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

import requests
from astropy.io import ascii
from astropy.table import Table
from astropy.units import UnitsWarning

warnings.filterwarnings("ignore", category=UnitsWarning)

BASE_URL = "https://svo2.cab.inta-csic.es/theory/newov2/colors.php"
MODELS_URL = "https://svo2.cab.inta-csic.es/theory/newov2/index.php"
FPS_URL = "https://svo2.cab.inta-csic.es/theory/fps/fps.php"

# The SVO logs should show this tool rather than an anonymous "python"
USER_AGENT = "colors/1.0 (SVO)"

TIMEOUT = 300   # seconds before giving up on a silent server

# Columns that arrive as text from the VOTable and are wanted as numbers. The fid is the
# spectrum's identifier, so it is kept as a whole number: it is a label, not a measurement.
INTEGER_COLUMNS = ("fid",)
NUMERIC_COLUMNS = ("teff", "logg", "meta", "alpha")


## 2. Building the URL

# The service expects the colors as color[1], color[2]... each holding a comma-separated
# pair, and the ranges as restri[teff]=2600/4000. The brackets and the slashes are part of
# that syntax, so urlencode is told to leave them alone: escaping them, which is what it
# would do by default, produces a URL the server does not understand.

def build_url(model, colors, ranges=None):
    """Build the query URL. It can be pasted into a browser as it is."""
    query = [("model", model)]

    for i, (first, second) in enumerate(colors, start=1):
        query.append((f"color[{i}]", f"{first},{second}"))

    for name, (low, high) in (ranges or {}).items():
        query.append((f"restri[{name}]", f"{low}/{high}"))

    return f"{BASE_URL}?{urlencode(query, safe='[]/,')}"


## 3. One query

# The service returns a VOTable. Its columns are named color_1, color_2, etc., 
# while the meaning of each colour is specified separately in a PARAM named colordef_N. 
# Reading these definitions allows us to replace the generic column names with the corresponding filter combinations.

def query_svo(url):
    """Send the request and return the column names, the rows and the PARAMs."""
    response = requests.get(url, timeout=TIMEOUT, headers={"User-Agent": USER_AGENT})
    response.raise_for_status()
    return read_votable(response.text)


def read_votable(text):
    """Read a VOTable into column names, rows of text and a dictionary of PARAMs."""
    root = ET.fromstring(text)

    # endswith is used because the VOTable carries a namespace, so the tag arrives as
    # "{http://www.ivoa.net/xml/VOTable/v1.1}INFO"
    for element in root.iter():
        if element.tag.endswith("INFO") and element.get("name") == "QUERY_STATUS":
            if element.get("value") != "OK":
                raise RuntimeError(
                    f"The SVO answered {element.get('value')}: {(element.text or '').strip()}"
                )

    params = {e.get("name"): e.get("value")
              for e in root.iter() if e.tag.endswith("PARAM")}
    names = [e.get("name") for e in root.iter() if e.tag.endswith("FIELD")]
    rows = [[td.text for td in row if td.tag.endswith("TD")]
            for row in root.iter() if row.tag.endswith("TR")]

    return names, rows, params


## 4. What is available

# Two lookups, so the model and the filters can be chosen without leaving the notebook.
# The model identifiers are the ones the server itself uses in its URLs, and the filter ones
# are those of the Filter Profile Service. 

def list_models():
    """Return the identifiers of every collection the server offers."""
    html = requests.get(MODELS_URL, timeout=TIMEOUT, headers={"User-Agent": USER_AGENT}).text

    # The page links each collection as index.php?models=<identifier>, so the identifiers can
    # be read off the links themselves. Order is preserved and repeats dropped, since the same
    # collection is linked more than once.
    found = re.findall(r"models=([A-Za-z0-9_.\-]+)", html)
    return list(dict.fromkeys(found))


def list_filters(facility=None, instrument=None, phot_system=None, band=None):
    """Return the filters matching the given criteria, as a table.

    The service publishes over six thousand filters, so at least one criterion is required.
    Facility is the observatory or mission ("2MASS", "JWST", "GAIA"), phot_system the
    photometric system ("SDSS", "Johnson").
    """
    criteria = {"Facility": facility, "Instrument": instrument,
                "PhotSystem": phot_system, "Band": band}
    query = {k: v for k, v in criteria.items() if v}
    if not query:
        raise ValueError("Give at least one criterion, for instance facility='2MASS'.")

    url = f"{FPS_URL}?{urlencode(query)}"
    names, rows, _ = read_votable(
        requests.get(url, timeout=TIMEOUT, headers={"User-Agent": USER_AGENT}).text)

    if not rows:
        print("No filter matches those criteria.")
        return Table(names=names)

    table = Table(rows=rows, names=names, dtype=[str] * len(names))
    wanted = [c for c in ("filterID", "WavelengthEff", "WidthEff", "ZeroPoint") if c in names]
    return table[wanted]


_filter_index = None


def filter_index():
    """ The whole FPS catalogue, fetched once and then kept in memory.    
    """
    global _filter_index
    if _filter_index is None:
        url = f"{FPS_URL}?FORMAT=metadata"
        names, rows, _ = read_votable(
            requests.get(url, timeout=TIMEOUT, headers={"User-Agent": USER_AGENT}).text)
        _filter_index = Table(rows=rows, names=names, dtype=[str] * len(names))
    return _filter_index


def list_facilities():
    """ Return the observatories and missions in the catalogue, together with the number of filters available for each one.
    """
    index = filter_index()
    counts = {}
    for name in index["Facility"]:
        name = (name or "").strip()
        if name:
            counts[name] = counts.get(name, 0) + 1
    return sorted(counts.items(), key=lambda pair: -pair[1])


## 5. The table

# The colordef_N declarations become the column names, so a column reads
# "2MASS/2MASS.H - 2MASS/2MASS.J" rather than "color_1". The photometric system of each
# filter is kept in the table's metadata, since the same filter calibrated in Vega, AB or
# ST gives three different numbers.

def get_colors(model, colors, ranges=None):
    """Query the service and return one table of synthetic colors."""
    url = build_url(model, colors, ranges)
    names, rows, params = query_svo(url)

    if not rows:
        print("No model matches the given ranges.")
        return Table(names=names)

    # color_1 -> "2MASS/2MASS.H - 2MASS/2MASS.J"
    labels = [params.get(name.replace("color_", "colordef_"), name) for name in names]

    table = Table(rows=rows, names=labels, dtype=[str] * len(labels))
    for column, name in zip(labels, names):
        if name in INTEGER_COLUMNS:
            table[column] = [int(value) for value in table[column]]
        elif name in NUMERIC_COLUMNS or name.startswith("color_"):
            table[column] = [float(value) for value in table[column]]

    return describe(table, params, url)


def describe(table, params, url):
    """Attach units and record where the numbers came from."""
    for column in table.colnames:
        if " - " in column:
            table[column].unit = "mag"
    if "teff" in table.colnames:
        table["teff"].unit = "K"

    table.meta["url"] = url
    table.meta["systems"] = {
        params[f"filter_{i}"]: params.get(f"magsys_{i}")
        for i in range(1, 99) if f"filter_{i}" in params
    }
    return table


## 6. Saving the table

# The extension picks the format: .vot for a VOTable, anything else for CSV. 

# comments=True, this information is enabled and currently stored in table.meta.

def save_table(table, path, comments=False):
    """Write the table to disk. A .vot extension gives a VOTable, otherwise CSV."""
    if str(path).endswith(".vot"):
        # A VOTable gives every column an ID as well as a name, and an XML ID cannot hold
        # the slashes and spaces a colour name carries. astropy invents one and says so,
        # which is harmless but noisy, so a valid ID is supplied here instead. The name,
        # which is what a reader actually sees, is left untouched.
        copy = table.copy()
        for i, column in enumerate(copy.colnames, start=1):
            copy[column].meta["ID"] = f"col{i}"
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            copy.write(path, format="votable", overwrite=True)
        return path

    with open(path, "w", encoding="utf-8") as handle:
        if comments:
            lines = [f"# url: {table.meta.get('url', '')}"]
            for name, system in (table.meta.get("systems") or {}).items():
                lines.append(f"# filter: {name}, magsys: {system}")
            handle.write("\n".join(lines) + "\n")
        ascii.write(table, handle, format="csv")
    return path


## 2. The query

The user now needs to select:

I) A collection of theoretical models. A full description of each collection, including references and parameter ranges, is available on the [models page](https://svo2.cab.inta-csic.es/theory/newov2/). The physical parameters and their ranges vary between collections.

II) The photometric filters used to construct the colours. The complete list of available filters is provided by the [Filter Profile Service](https://svo2.cab.inta-csic.es/theory/fps/).


| Field | Meaning | May be empty |
|---|---|---|
| `model` | collection identifier, as it appears in the server URL | no |
| `colors` | pairs of filters, one per colour wanted | no |
| `ranges` | ranges for the physical parameters | yes, takes the whole grid |

The parameter names in `ranges` differ from one collection to another. BT-Settl and Kurucz use `teff`, `logg` and `meta`; the Koester white dwarfs have no metallicity; exoplanet atmosphere grids use names of their own. Both ends of a range are included.


In [8]:
model = "cond00"

colors = [
    ("2MASS/2MASS.H", "2MASS/2MASS.J"),     # H - J
    ("2MASS/2MASS.Ks", "2MASS/2MASS.J"),    # Ks - J
]

ranges = {
    "teff": (100, 4000),
    "logg": (2.5, 6.0),
}

## 3. The result

The URL is printed first.

In [9]:
print(build_url(model, colors, ranges))

table = get_colors(model, colors, ranges)
table

https://svo2.cab.inta-csic.es/theory/newov2/colors.php?model=cond00&color[1]=2MASS/2MASS.H,2MASS/2MASS.J&color[2]=2MASS/2MASS.Ks,2MASS/2MASS.J&restri[teff]=100/4000&restri[logg]=2.5/6.0


model,fid,teff,logg,meta,2MASS/2MASS.H - 2MASS/2MASS.J,2MASS/2MASS.Ks - 2MASS/2MASS.J
,,K,,,mag,mag
str6,int64,float64,float64,float64,float64,float64
cond00,1,100.0,2.5,0.0,-6.5659465358981,15.713684085636
cond00,2,100.0,3.0,0.0,-5.856500482361,18.409870015906
cond00,3,100.0,3.5,0.0,-5.4797627399585,18.117659821848
cond00,5,100.0,4.5,0.0,-4.5436240634496,18.368667484054
cond00,6,100.0,5.0,0.0,-4.415744194798,18.437221038292
cond00,7,100.0,5.5,0.0,-4.4152403236797,18.293154352093
cond00,8,100.0,6.0,0.0,-4.4237051340951,18.539803354541
cond00,9,200.0,2.5,0.0,-1.8615141264666,11.881650321139


## 4. Saving it


In [10]:
save_table(table, "colors.csv")
save_table(table, "colors.vot")

'colors.vot'